In [1]:
import duckdb
from dotenv import load_dotenv
import os
import s3fs
import polars as pl
import boto3
load_dotenv()

pl.Config.set_fmt_str_lengths(50)

polars.config.Config

In [3]:

s3 = s3fs.S3FileSystem(
    key=os.getenv("S3_ACCESS_KEY"),
    secret=os.getenv("S3_SECRET_KEY"),
    token=os.getenv("AWS_SESSION_TOKEN"),  # optional but important if using temp creds
    client_kwargs={
        "region_name": "ca-central-1"
    }
)

In [4]:
# s3 setup using boto3
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv("S3_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("S3_SECRET_KEY"),
    aws_session_token=os.getenv("AWS_SESSION_TOKEN"),  # optional but important if using temp creds
    region_name='ca-central-1'
)

In [5]:
# scan s3 bucket using duckdb

con = duckdb.connect()

bucket = os.getenv("S3_BUCKET")

snp500 = con.execute(f"""
SELECT *
FROM read_csv_auto('s3://{bucket}/post-earnings-forecast/**/*.csv')

""").df()
snp500

,#,Company,Symbol,Weight,Price,Chg,% Chg
0,1,NVIDIA Corp,NVDA,7.69%,215.33,-4.18,-1.90%
1,2,Apple Inc,AAPL,6.69%,308.82,3.83,1.26%
2,3,Microsoft Corp,MSFT,4.58%,418.57,-0.52,-0.12%
3,4,Amazon.com Inc,AMZN,4.22%,266.32,-2.14,-0.80%
4,5,Alphabet Inc,GOOGL,3.53%,382.97,-4.69,-1.21%
...,...,...,...,...,...,...,...
498,499,Pool Corp,POOL,0.01%,184.64,2.95,1.62%
499,500,Conagra Brands Inc,CAG,0.01%,13.56,0.18,1.35%
500,501,Campbell's Company/The,CPB,0.01%,20.58,0.53,2.64%
501,502,News Corp,NWS,0.01%,29.68,-0.40,-1.33%


# Earnings


In [6]:
#  delta table for earnings
DELTA_EARNINGS = f"s3://{os.getenv('S3_BUCKET')}/post-earnings-forecast/earnings_delta/"


# storage options for delta lake
storage_options = {
    "AWS_ACCESS_KEY_ID": os.getenv("AWS_ACCESS_KEY_ID"),
    "AWS_SECRET_ACCESS_KEY": os.getenv("AWS_SECRET_ACCESS_KEY"),
    "AWS_REGION": "ca-central-1",
}


delta_earnings_df = pl.scan_delta(DELTA_EARNINGS, storage_options=storage_options).collect()
delta_earnings_df


delta_earnings_df




fiscalDateEnding,reportedDate,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,symbol,fiscal_month,fiscal_year,av_quarter,time_from
date,date,str,str,str,str,str,str,i8,i32,str,str
2015-09-30,2015-11-06,"""1.85""","""1.85""","""0""","""0""","""post-market""","""BRK-B""",9,2015,"""2015Q3""","""20151106T0000"""
2019-03-31,2019-05-04,"""2.26""","""2.29""","""-0.03""","""-1.31""","""pre-market""","""BRK-B""",3,2019,"""2019Q1""","""20190504T0000"""
2017-09-30,2017-11-03,"""1.4""","""1.58""","""-0.18""","""-11.3924""","""post-market""","""BRK-B""",9,2017,"""2017Q3""","""20171103T0000"""
2016-12-31,2017-02-25,"""1.78""","""1.83""","""-0.05""","""-2.7322""","""pre-market""","""BRK-B""",12,2016,"""2016Q4""","""20170225T0000"""
2023-06-30,2023-08-05,"""4.62""","""3.87""","""0.75""","""19.3798""","""pre-market""","""BRK-B""",6,2023,"""2023Q2""","""20230805T0000"""
…,…,…,…,…,…,…,…,…,…,…,…
2014-09-30,2014-10-22,"""0.02""","""0.02""","""0""","""0""","""post-market""","""FTNT""",9,2014,"""2014Q3""","""20141022T0000"""
2025-09-30,2025-11-05,"""0.74""","""0.63""","""0.11""","""17.4603""","""post-market""","""FTNT""",9,2025,"""2025Q3""","""20251105T0000"""
2025-03-31,2025-05-07,"""0.58""","""0.53""","""0.05""","""9.434""","""post-market""","""FTNT""",3,2025,"""2025Q1""","""20250507T0000"""


In [80]:
df_earning = delta_earnings_df.filter(
    (pl.col('symbol') == 'NVDA') & 
    (pl.col('av_quarter') == '2014Q1')
)
df_earning_selected = df_earning.select(
    'reportedDate',
    'symbol',
    'av_quarter',
    'reportedEPS',
    'estimatedEPS',
    'surprise',
    'surprisePercentage'
)
df_earning_selected


reportedDate,symbol,av_quarter,reportedEPS,estimatedEPS,surprise,surprisePercentage
date,str,str,str,str,str,str
2014-02-12,"""NVDA""","""2014Q1""","""0.006""","""0.005""","""0.001""","""20"""


In [81]:
df_earning_selected.to_pandas().to_clipboard()

In [82]:
reportedDate = df_earning_selected['reportedDate'].to_list()
reportedDate

[datetime.date(2014, 2, 12)]

In [76]:
from datetime import timedelta

date_minus_10 = df_earning['reportedDate'] - timedelta(days=10)
date_plus_10 = df_earning['reportedDate'] + timedelta(days=10)

print(date_minus_10)
print(date_plus_10)

shape: (1,)
Series: 'reportedDate' [date]
[
	2014-02-02
]
shape: (1,)
Series: 'reportedDate' [date]
[
	2014-02-22
]


# OHLVC

In [77]:
from datetime import timedelta

DELTA_OHLVC = f"s3://{bucket}/post-earnings-forecast/ohlcv_delta/"
df_ohlvc = pl.read_delta(DELTA_OHLVC, storage_options=storage_options)

# Get reported date
reported_date = date_minus_10[0] + timedelta(days=10)

# Filter for symbol and get all data around reported date
result = df_ohlvc.filter(
    (pl.col('symbol') == 'NVDA')
).sort('date')

# Add row numbers
result_with_rn = result.with_columns(
    row_num=pl.int_range(pl.len())
)

# Find the row index of reported date (or closest date)
reported_row = result_with_rn.filter(pl.col('date') <= reported_date).select(pl.col('row_num')).max().item()

# Get 10 rows before and 10 rows after reported date
result_final = result_with_rn.filter(
    (pl.col('row_num') >= reported_row - 10) & 
    (pl.col('row_num') <= reported_row + 10)
).drop('row_num')

result_final

symbol,date,open,high,low,close,volume
str,date,f64,f64,f64,f64,i64
"""NVDA""",2014-01-29,0.366536,0.368658,0.364179,0.364415,162944000
"""NVDA""",2014-01-30,0.366536,0.371251,0.365358,0.370544,202896000
"""NVDA""",2014-01-31,0.365358,0.372194,0.365122,0.370072,335348000
"""NVDA""",2014-02-03,0.372665,0.373136,0.363708,0.365122,431728000
"""NVDA""",2014-02-04,0.364887,0.367951,0.363001,0.367244,280692000
…,…,…,…,…,…,…
"""NVDA""",2014-02-21,0.441965,0.447622,0.436544,0.439372,451052000
"""NVDA""",2014-02-24,0.438429,0.449037,0.434422,0.445737,403112000
"""NVDA""",2014-02-25,0.445145,0.447039,0.438988,0.443724,242184000


In [79]:
result_final.to_pandas().to_clipboard()

In [84]:
# Step 1: Add t-labels to result_final
# Find the index of the reported date row (t0)
reported_row_idx = result_with_rn.filter(pl.col('date') <= reported_date).select(pl.col('row_num')).max().item()

# Add offset column relative to reported date
result_labeled = result_with_rn.filter(
    (pl.col('row_num') >= reported_row_idx - 10) & 
    (pl.col('row_num') <= reported_row_idx + 10)
).with_columns(
    t_offset=(pl.col('row_num') - reported_row_idx)
).drop('row_num')

# Step 2: Pivot each row into columns
cols_to_pivot = ['date', 'open', 'high', 'low', 'close', 'volume'] 

pivoted_dict = {}
for row in result_labeled.iter_rows(named=True):
    offset = row['t_offset']
    prefix = f"t{offset}" if offset <= 0 else f"t+{offset}"
    for col in cols_to_pivot:
        pivoted_dict[f"{prefix}_{col}"] = [row[col]]

df_pivoted = pl.DataFrame(pivoted_dict)

# Step 3: Join with df_earning_selected (cross join since both are single rows,
# or use a key if you're doing this for multiple symbols/quarters)
df_merged = pl.concat([df_earning_selected, df_pivoted], how='horizontal')

df_merged

reportedDate,symbol,av_quarter,reportedEPS,estimatedEPS,surprise,surprisePercentage,t-10_date,t-10_open,t-10_high,t-10_low,t-10_close,t-10_volume,t-9_date,t-9_open,t-9_high,t-9_low,t-9_close,t-9_volume,t-8_date,t-8_open,t-8_high,t-8_low,t-8_close,t-8_volume,t-7_date,t-7_open,t-7_high,t-7_low,t-7_close,t-7_volume,t-6_date,t-6_open,t-6_high,t-6_low,t-6_close,t-6_volume,…,t+4_volume,t+5_date,t+5_open,t+5_high,t+5_low,t+5_close,t+5_volume,t+6_date,t+6_open,t+6_high,t+6_low,t+6_close,t+6_volume,t+7_date,t+7_open,t+7_high,t+7_low,t+7_close,t+7_volume,t+8_date,t+8_open,t+8_high,t+8_low,t+8_close,t+8_volume,t+9_date,t+9_open,t+9_high,t+9_low,t+9_close,t+9_volume,t+10_date,t+10_open,t+10_high,t+10_low,t+10_close,t+10_volume
date,str,str,str,str,str,str,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,…,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64
2014-02-12,"""NVDA""","""2014Q1""","""0.006""","""0.005""","""0.001""","""20""",2014-01-29,0.366536,0.368658,0.364179,0.364415,162944000,2014-01-30,0.366536,0.371251,0.365358,0.370544,202896000,2014-01-31,0.365358,0.372194,0.365122,0.370072,335348000,2014-02-03,0.372665,0.373136,0.363708,0.365122,431728000,2014-02-04,0.364887,0.367951,0.363001,0.367244,280692000,…,450340000,2014-02-20,0.42853,0.443615,0.427823,0.442673,398088000,2014-02-21,0.441965,0.447622,0.436544,0.439372,451052000,2014-02-24,0.438429,0.449037,0.434422,0.445737,403112000,2014-02-25,0.445145,0.447039,0.438988,0.443724,242184000,2014-02-26,0.442777,0.449407,0.441593,0.443014,360884000,2014-02-27,0.443014,0.447513,0.436857,0.438041,388848000


In [85]:
df_merged.to_pandas().to_clipboard()

In [88]:
from datetime import timedelta

def build_earnings_ohlcv_dataset(symbol: str, delta_earnings_df: pl.DataFrame, df_ohlvc: pl.DataFrame) -> pl.DataFrame:
    """
    For a given symbol, loops over all quarters from 2014Q1 to 2023Q4,
    merges earnings data with OHLCV data (t-10 to t+10 around reported date),
    and returns a single wide DataFrame with one row per quarter.
    """
    
    # Filter earnings for this symbol
    df_symbol_earnings = delta_earnings_df.filter(pl.col('symbol') == symbol)
    
    # Filter OHLCV for this symbol once (more efficient)
    df_ohlvc_symbol = df_ohlvc.filter(pl.col('symbol') == symbol).sort('date')
    
    # Add row numbers to OHLCV once
    df_ohlvc_rn = df_ohlvc_symbol.with_columns(
        row_num=pl.int_range(pl.len())
    )
    
    cols_to_pivot = ['date', 'open', 'high', 'low', 'close', 'volume']
    
    all_rows = []
    
    # Generate all quarters from 2014Q1 to 2023Q4
    quarters = [f"{year}Q{q}" for year in range(2014, 2024) for q in range(1, 5)]
    
    for quarter in quarters:
        # Filter earnings for this quarter
        df_earning = df_symbol_earnings.filter(pl.col('av_quarter') == quarter)
        
        if df_earning.is_empty():
            print(f"No earnings data for {symbol} {quarter}, skipping.")
            continue
        
        df_earning_selected = df_earning.select(
            'reportedDate', 'symbol', 'av_quarter',
            'reportedEPS', 'estimatedEPS', 'surprise', 'surprisePercentage'
        )
        
        reported_date = df_earning['reportedDate'][0]
        
        # Find the row index of reported date (or closest date before it)
        reported_row_idx = df_ohlvc_rn.filter(
            pl.col('date') <= reported_date
        ).select(pl.col('row_num')).max().item()
        
        if reported_row_idx is None:
            print(f"No OHLCV data found around {symbol} {quarter}, skipping.")
            continue
        
        # Get t-10 to t+10 window
        result_labeled = df_ohlvc_rn.filter(
            (pl.col('row_num') >= reported_row_idx - 10) &
            (pl.col('row_num') <= reported_row_idx + 10)
        ).with_columns(
            t_offset=(pl.col('row_num') - reported_row_idx)
        ).drop('row_num')
        
        # Pivot rows into columns
        pivoted_dict = {}
        for row in result_labeled.iter_rows(named=True):
            offset = row['t_offset']
            prefix = f"t_" if offset == 0 else (f"t{offset}" if offset < 0 else f"t+{offset}")
            for col in cols_to_pivot:
                pivoted_dict[f"{prefix}_{col}"] = [row[col]]
        
        df_pivoted = pl.DataFrame(pivoted_dict)
        df_merged_row = pl.concat([df_earning_selected, df_pivoted], how='horizontal')
        all_rows.append(df_merged_row)
    
    if not all_rows:
        print(f"No data found for {symbol}.")
        return pl.DataFrame()
    
    return pl.concat(all_rows, how='diagonal')  # diagonal handles any missing t-columns across quarters


# Usage
df_merged_f = build_earnings_ohlcv_dataset('NVDA', delta_earnings_df, df_ohlvc)
df_merged_f

reportedDate,symbol,av_quarter,reportedEPS,estimatedEPS,surprise,surprisePercentage,t-10_date,t-10_open,t-10_high,t-10_low,t-10_close,t-10_volume,t-9_date,t-9_open,t-9_high,t-9_low,t-9_close,t-9_volume,t-8_date,t-8_open,t-8_high,t-8_low,t-8_close,t-8_volume,t-7_date,t-7_open,t-7_high,t-7_low,t-7_close,t-7_volume,t-6_date,t-6_open,t-6_high,t-6_low,t-6_close,t-6_volume,…,t+4_volume,t+5_date,t+5_open,t+5_high,t+5_low,t+5_close,t+5_volume,t+6_date,t+6_open,t+6_high,t+6_low,t+6_close,t+6_volume,t+7_date,t+7_open,t+7_high,t+7_low,t+7_close,t+7_volume,t+8_date,t+8_open,t+8_high,t+8_low,t+8_close,t+8_volume,t+9_date,t+9_open,t+9_high,t+9_low,t+9_close,t+9_volume,t+10_date,t+10_open,t+10_high,t+10_low,t+10_close,t+10_volume
date,str,str,str,str,str,str,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,…,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64,date,f64,f64,f64,f64,i64
2014-02-12,"""NVDA""","""2014Q1""","""0.006""","""0.005""","""0.001""","""20""",2014-01-29,0.366536,0.368658,0.364179,0.364415,162944000,2014-01-30,0.366536,0.371251,0.365358,0.370544,202896000,2014-01-31,0.365358,0.372194,0.365122,0.370072,335348000,2014-02-03,0.372665,0.373136,0.363708,0.365122,431728000,2014-02-04,0.364887,0.367951,0.363001,0.367244,280692000,…,450340000,2014-02-20,0.42853,0.443615,0.427823,0.442673,398088000,2014-02-21,0.441965,0.447622,0.436544,0.439372,451052000,2014-02-24,0.438429,0.449037,0.434422,0.445737,403112000,2014-02-25,0.445145,0.447039,0.438988,0.443724,242184000,2014-02-26,0.442777,0.449407,0.441593,0.443014,360884000,2014-02-27,0.443014,0.447513,0.436857,0.438041,388848000
2014-05-06,"""NVDA""","""2014Q2""","""0.006""","""0.004""","""0.002""","""50""",2014-04-22,0.443724,0.449644,0.441356,0.446802,255120000,2014-04-23,0.447513,0.453195,0.446802,0.452011,261580000,2014-04-24,0.45509,0.460772,0.45509,0.456037,391492000,2014-04-25,0.45509,0.459352,0.44254,0.443487,313600000,2014-04-28,0.447039,0.449407,0.436147,0.441593,226788000,…,299856000,2014-05-13,0.434016,0.44112,0.431412,0.432832,221104000,2014-05-14,0.433069,0.434727,0.428334,0.42857,214320000,2014-05-15,0.427623,0.429991,0.42194,0.426203,325896000,2014-05-16,0.422888,0.428333,0.422888,0.425255,283960000,2014-05-19,0.429754,0.441356,0.42786,0.438989,360076000,2014-05-20,0.439107,0.440297,0.431734,0.433874,256200000
2014-08-07,"""NVDA""","""2014Q3""","""0.006""","""0.005""","""0.001""","""20""",2014-07-24,0.432447,0.433636,0.42769,0.430782,254576000,2014-07-25,0.429355,0.430306,0.422457,0.42317,266132000,2014-07-28,0.425073,0.425073,0.414369,0.421505,330352000,2014-07-29,0.421743,0.428641,0.421505,0.422932,217972000,2014-07-30,0.425549,0.431258,0.42436,0.430068,246184000,…,256596000,2014-08-14,0.453856,0.453856,0.445292,0.447195,255992000,2014-08-15,0.448622,0.455521,0.444341,0.452904,369796000,2014-08-18,0.454807,0.459564,0.450287,0.459089,284076000,2014-08-19,0.459208,0.465181,0.457536,0.462792,248260000,2014-08-20,0.461119,0.46327,0.45873,0.459925,221468000,2014-08-21,0.45873,0.461119,0.454907,0.455624,272796000
2014-11-06,"""NVDA""","""2014Q4""","""0.008""","""0.007""","""0.001""","""14.2857""",2014-10-23,0.434838,0.440094,0.431732,0.436988,214476000,2014-10-24,0.439616,0.444156,0.436988,0.441528,210156000,2014-10-27,0.441289,0.443439,0.43651,0.441767,145092000,2014-10-28,0.440572,0.452757,0.440094,0.452279,192580000,2014-10-29,0.450368,0.452996,0.445351,0.449412,168340000,…,204584000,2014-11-13,0.471154,0.471871,0.46327,0.467092,225328000,2014-11-14,0.46757,0.472827,0.463031,0.472827,160248000,2014-11-17,0.470676,0.475694,0.46757,0.470676,158732000,2014-11-18,0.470915,0.4831,0.46972,0.481905,207772000,2014-11-19,0.482265,0.483705,0.475067,0.480106,240300000,2014-11-20,0.476747,0.488504,0.475307,0.488024,220968000
2015-02-11,"""NVDA""","""2015Q1""","""0.009""","""0.007""","""0.002""","""28.5714""",2015-01-28,0.476747,0

In [95]:
count = 0
for i in range(len(df_merged_f) - 1):
    t10_plus = df_merged_f[i, 't+10_date']
    next_t10_minus = df_merged_f[i+1, 't-10_date']
    if t10_plus >= next_t10_minus:
        print(f"Overlap between row {i} ({df_merged_f[i,'av_quarter']}) and row {i+1} ({df_merged_f[i+1,'av_quarter']})")
        count=count+1
if count == 0:
        print(f"There is no overlap.")  

There is no overlap.


In [89]:
df_merged_f.write_clipboard()